# 03c — Representaciones temporales y selección de capas wav2vec

Este notebook evalúa, sin abrir los test finales, cuatro extensiones controladas de las referencias de Sprint 3:

1. **Statistics pooling:** `mean` frente a `mean + std`.
2. **Capas wav2vec:** última capa, promedio uniforme y mezcla escalar aprendida.
3. **Attentive statistics pooling:** selección temporal aprendida sobre `z_t` congelado.
4. **Elastic Net eGeMAPS:** regularización interpretable agregada por familias acústicas.

La comparación principal es `speaker-independent · 8 emociones`. Los resultados secundarios se habilitan mediante flags. Toda transformación aprendida se ajusta dentro de outer train.

## 1. Configuración, artefactos y alcance

Los flags están desactivados por defecto. La extracción temporal se ejecuta una sola vez; los experimentos posteriores cargan los artefactos congelados.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import confusion_matrix

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.contracts import (
    PROTOCOL_INDEPENDENT,
    TARGET_EMOTION_ORIGINAL,
    TARGET_EMOTION_ORIGINAL_EVAL_QUADRANT,
    TARGET_EMOTION_QUADRANT,
)
from src.evaluation.reporting import summarize_cv_results
from src.experiments.cross_validation import run_cv
from src.experiments.temporal_representation import (
    run_attentive_pooling_cv,
    run_egemaps_elastic_net,
    run_layer_mixture_grid,
    run_static_pooling_grid,
    save_attention_weights,
)
from src.features.feature_store import load_representation
from src.features.wav2vec_temporal import (
    extract_or_load_temporal_features,
    load_layer_statistics,
)
from src.models.linear_probe import build_linear_probe
from src.utils.config import get_config, resolve_path
from src.utils.io import load_metadata, load_splits
from src.utils.reproducibility import set_global_seed

cfg = get_config()
set_global_seed(int(cfg.seed))

metadata = load_metadata(cfg.paths.metadata)
splits = load_splits(cfg.paths.splits)
egemaps = load_representation("egemaps", cfg.paths.features_dir)
wav2vec_mean = load_representation("wav2vec", cfg.paths.features_dir)

REPORTS_DIR = resolve_path(cfg.paths.reports_dir)
FIGURES_DIR = resolve_path(cfg.paths.figures_dir)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

STATS_PATH = resolve_path(cfg.temporal_representation.layer_statistics_path)
SEQUENCES_DIR = resolve_path(cfg.temporal_representation.sequences_dir)
RESULTS_PATH = REPORTS_DIR / "temporal_representation_results.csv"
OOF_PATH = REPORTS_DIR / "temporal_oof_predictions.csv"
LAYER_WEIGHTS_PATH = REPORTS_DIR / "layer_mixture_weights.csv"
ATTENTION_SEED_PATH = REPORTS_DIR / "attention_seed_results.csv"
ATTENTION_WEIGHTS_PATH = REPORTS_DIR / "attention_weights_selected.npz"
ELASTIC_PATH = REPORTS_DIR / "egemaps_elastic_net_results.csv"
ELASTIC_FAMILY_PATH = REPORTS_DIR / "egemaps_elastic_net_family_importance.csv"

RUN_FEATURE_EXTRACTION = True
RUN_STATIC_POOLING = True
RUN_LAYER_MIXTURE = True
RUN_ATTENTIVE_POOLING = True
RUN_EGEMAPS_ELASTIC_NET = True

RUN_SECONDARY_TARGETS = True

TARGETS = [TARGET_EMOTION_ORIGINAL]
if RUN_SECONDARY_TARGETS:
    TARGETS += [TARGET_EMOTION_QUADRANT, TARGET_EMOTION_ORIGINAL_EVAL_QUADRANT]

PROTOCOLS = [PROTOCOL_INDEPENDENT]

print(f"Metadata: {len(metadata)} audios")
print(f"Targets: {TARGETS}")
print(f"Protocolos: {PROTOCOLS}")

In [ ]:
def load_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def upsert_rows(existing: pd.DataFrame, new: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    if existing.empty:
        return new.copy().reset_index(drop=True)
    if new.empty:
        return existing.copy().reset_index(drop=True)
    common_keys = [key for key in keys if key in existing.columns and key in new.columns]
    if len(common_keys) != len(keys):
        missing = set(keys) - set(common_keys)
        raise KeyError(f"Faltan claves para consolidar resultados: {sorted(missing)}")
    old_index = pd.MultiIndex.from_frame(existing[common_keys].fillna("<NA>"))
    new_index = pd.MultiIndex.from_frame(new[common_keys].fillna("<NA>"))
    retained = existing.loc[~old_index.isin(new_index)]
    return pd.concat([retained, new], ignore_index=True, sort=False)


def save_results(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


RESULT_KEYS = [
    "representation", "protocol", "target", "model", "refinement",
    "fold", "seed", "result_type",
]
PREDICTION_KEYS = [
    "file_id", "protocol", "target", "model", "refinement", "fold",
]

results = load_csv(RESULTS_PATH)
oof_predictions = load_csv(OOF_PATH)
layer_weights = load_csv(LAYER_WEIGHTS_PATH)
elastic_family = load_csv(ELASTIC_FAMILY_PATH)

print("Resultados temporales cargados:", len(results))
print("Predicciones OOF cargadas:", len(oof_predictions))

## 2. Extracción congelada de hidden states

Se guarda `mean/std` por capa en un único NPZ y una secuencia ragged de última capa por audio. El encoder permanece en `eval()` y sin gradientes; no se usan targets ni folds.

In [ ]:
if RUN_FEATURE_EXTRACTION:
    temporal_stats = extract_or_load_temporal_features(
        metadata=metadata,
        statistics_path=STATS_PATH,
        sequences_dir=SEQUENCES_DIR,
        path_col="file_path_trimmed",
        model_name=cfg.features.wav2vec.model_name,
        revision=cfg.features.wav2vec.revision,
        target_sample_rate=int(cfg.audio.target_sample_rate),
        include_feature_projection=bool(
            cfg.temporal_representation.include_feature_projection
        ),
        sequence_dtype=str(cfg.temporal_representation.sequence_dtype),
        overwrite=False,
    )
else:
    temporal_stats = (
        load_layer_statistics(STATS_PATH, expected_file_ids=metadata["file_id"])
        if STATS_PATH.exists()
        else None
    )

if temporal_stats is None:
    print("No hay layer statistics. Active RUN_FEATURE_EXTRACTION una sola vez.")
else:
    print(
        f"Layer statistics: {temporal_stats.n_files} audios · "
        f"{temporal_stats.n_layers} capas · {temporal_stats.hidden_size} dimensiones"
    )

## 3. Poolings estáticos: media frente a media + desvío

Este bloque pregunta si la dispersión temporal de `z_t` contiene información que desaparece al conservar únicamente la media.

In [ ]:
if RUN_STATIC_POOLING:
    if temporal_stats is None:
        raise RuntimeError("Primero debe extraerse wav2vec_layer_statistics.npz")
    output = run_static_pooling_grid(
        stats=temporal_stats,
        metadata=metadata,
        splits=splits,
        logistic_regression_config=dict(cfg.models.logistic_regression),
        seed=int(cfg.seed),
        layer_strategies=list(cfg.temporal_representation.static_pooling.strategies),
        poolings=list(cfg.temporal_representation.static_pooling.poolings),
        protocols=PROTOCOLS,
        targets=TARGETS,
        n_folds=int(cfg.splits.cv_folds),
    )
    static_results = output["fold_results"].copy()
    static_results["seed"] = -1
    static_results["result_type"] = "deterministic"
    results = upsert_rows(results, static_results, RESULT_KEYS)
    oof_predictions = upsert_rows(
        oof_predictions, output["predictions"], PREDICTION_KEYS
    )
    save_results(results, RESULTS_PATH)
    save_results(oof_predictions, OOF_PATH)

## 4. Mezcla aprendida de capas

Se aprenden pocos pesos escalares `softmax`, uno por capa, y una cabeza lineal. Las tres semillas se agregan por fold; la variabilidad entre semillas se conserva separada de la variabilidad entre actores.

In [ ]:
if RUN_LAYER_MIXTURE:
    if temporal_stats is None:
        raise RuntimeError("Primero debe extraerse wav2vec_layer_statistics.npz")
    layer_cfg = cfg.temporal_representation.layer_mixture
    output = run_layer_mixture_grid(
        stats=temporal_stats,
        metadata=metadata,
        splits=splits,
        seeds=list(layer_cfg.seeds),
        poolings=list(layer_cfg.poolings),
        protocols=PROTOCOLS,
        targets=TARGETS,
        learning_rate=float(layer_cfg.learning_rate),
        weight_decay=float(layer_cfg.weight_decay),
        max_epochs=int(layer_cfg.max_epochs),
        patience=int(layer_cfg.patience),
        inner_folds=3,
        n_folds=int(cfg.splits.cv_folds),
    )
    results = upsert_rows(results, output["fold_results"], RESULT_KEYS)
    oof_predictions = upsert_rows(
        oof_predictions, output["predictions"], PREDICTION_KEYS
    )
    layer_weights = upsert_rows(
        layer_weights,
        output["layer_weights"],
        ["protocol", "target", "pooling", "fold", "seed", "layer"],
    )
    save_results(results, RESULTS_PATH)
    save_results(oof_predictions, OOF_PATH)
    save_results(layer_weights, LAYER_WEIGHTS_PATH)

## 5. Attentive statistics pooling

La atención asigna un peso a cada frame y construye media y desvío ponderados. Early stopping se realiza sobre una partición interna de outer train; las probabilidades de tres semillas se promedian para cada muestra OOF.

In [ ]:
if RUN_ATTENTIVE_POOLING:
    att_cfg = cfg.temporal_representation.attentive_pooling
    output = run_attentive_pooling_cv(
        metadata=metadata,
        splits=splits,
        sequences_dir=SEQUENCES_DIR,
        seeds=list(att_cfg.seeds),
        protocols=PROTOCOLS,
        targets=TARGETS,
        input_dim=int(cfg.features.wav2vec.embedding_dim),
        attention_hidden_dim=int(att_cfg.attention_hidden_dim),
        dropout=float(att_cfg.dropout),
        learning_rate=float(att_cfg.learning_rate),
        weight_decay=float(att_cfg.weight_decay),
        max_epochs=int(att_cfg.max_epochs),
        patience=int(att_cfg.patience),
        batch_size=int(att_cfg.batch_size),
        inner_folds=3,
        n_folds=int(cfg.splits.cv_folds),
    )
    results = upsert_rows(results, output["fold_results"], RESULT_KEYS)
    oof_predictions = upsert_rows(
        oof_predictions, output["predictions"], PREDICTION_KEYS
    )
    save_results(results, RESULTS_PATH)
    save_results(oof_predictions, OOF_PATH)
    seed_rows = output["fold_results"].query("result_type == 'seed'")
    save_results(seed_rows, ATTENTION_SEED_PATH)
    save_attention_weights(output["attention_weights"], ATTENTION_WEIGHTS_PATH)

## 6. Elastic Net sobre eGeMAPS

Se compara la referencia L2 con una búsqueda interna pequeña de `C` y `l1_ratio`. La interpretación permanece agregada por familias acústicas.

In [ ]:
if RUN_EGEMAPS_ELASTIC_NET:
    # Referencia L2 con los mismos folds y targets.
    l2_frames = []
    l2_predictions = []
    for protocol in PROTOCOLS:
        for target in [item for item in TARGETS if item != TARGET_EMOTION_ORIGINAL_EVAL_QUADRANT]:
            output_l2 = run_cv(
                pipeline_factory=lambda: build_linear_probe(
                    params=dict(cfg.models.logistic_regression), seed=int(cfg.seed)
                ),
                representation=egemaps,
                metadata=metadata,
                splits=splits,
                target_col=target,
                representation_name="egemaps",
                model_name="logistic_regression",
                refinement="l2_reference",
                n_folds=int(cfg.splits.cv_folds),
            )
            fold_frame = output_l2["fold_results"].copy()
            fold_frame["seed"] = -1
            fold_frame["result_type"] = "deterministic"
            l2_frames.append(fold_frame)
            pred = output_l2["predictions"].copy()
            pred["protocol"] = protocol
            pred["target"] = target
            pred["model"] = "logistic_regression"
            pred["refinement"] = "l2_reference"
            l2_predictions.append(pred)

    elastic_cfg = cfg.temporal_representation.elastic_net
    output_en = run_egemaps_elastic_net(
        egemaps=egemaps,
        metadata=metadata,
        splits=splits,
        C_values=list(elastic_cfg.C_values),
        l1_ratios=list(elastic_cfg.l1_ratios),
        seed=int(cfg.seed),
        protocols=PROTOCOLS,
        targets=[item for item in TARGETS if item != TARGET_EMOTION_ORIGINAL_EVAL_QUADRANT],
        inner_folds=int(elastic_cfg.inner_folds),
        max_iter=int(elastic_cfg.max_iter),
        n_jobs=int(elastic_cfg.n_jobs),
        n_folds=int(cfg.splits.cv_folds),
    )
    combined = pd.concat([*l2_frames, output_en["fold_results"]], ignore_index=True)
    results = upsert_rows(results, combined, RESULT_KEYS)
    predictions_en = pd.concat(
        [*l2_predictions, output_en["predictions"]], ignore_index=True, sort=False
    )
    oof_predictions = upsert_rows(
        oof_predictions, predictions_en, PREDICTION_KEYS
    )
    elastic_family = output_en["family_importance"].copy()
    save_results(results, RESULTS_PATH)
    save_results(oof_predictions, OOF_PATH)
    save_results(output_en["fold_results"], ELASTIC_PATH)
    save_results(elastic_family, ELASTIC_FAMILY_PATH)

# Presentación de resultados

Las siguientes figuras cargan únicamente resultados persistidos. Las tablas completas permanecen en CSV; el notebook muestra medias, desvíos, curvas por fold, matrices normalizadas y visualizaciones de capas/atención.

In [ ]:
results = load_csv(RESULTS_PATH)
oof_predictions = load_csv(OOF_PATH)
layer_weights = load_csv(LAYER_WEIGHTS_PATH)
elastic_family = load_csv(ELASTIC_FAMILY_PATH)

if results.empty:
    print("Todavía no hay resultados temporales. Active los bloques de ejecución en orden.")
else:
    analysis_results = results.copy()
    if "result_type" in analysis_results:
        analysis_results = analysis_results.loc[
            ~analysis_results["result_type"].eq("seed")
        ]
    summary = summarize_cv_results(analysis_results)
    display(
        summary.loc[
            (summary["protocol"] == PROTOCOL_INDEPENDENT)
            & (summary["target"] == TARGET_EMOTION_ORIGINAL),
            ["representation", "model", "refinement", "macro_f1_mean", "macro_f1_std"],
        ].head(10).round(3)
    )

## Figura 1 — Pooling estático

La media representa rendimiento; la barra horizontal muestra el desvío entre outer folds. La comparación directa es `last_mean` frente a `last_mean_std`.

In [ ]:
if not results.empty:
    static = results.loc[
        (results["protocol"] == PROTOCOL_INDEPENDENT)
        & (results["target"] == TARGET_EMOTION_ORIGINAL)
        & (results["model"] == "logistic_regression")
        & (results["representation"] == "wav2vec_temporal")
    ]
    if static.empty:
        print("Sin resultados de pooling estático.")
    else:
        plot_data = (
            static.groupby("refinement", observed=True)["macro_f1"]
            .agg(["mean", "std"])
            .sort_values("mean")
        )
        fig, ax = plt.subplots(figsize=(8, 4.5))
        y = np.arange(len(plot_data))
        ax.errorbar(plot_data["mean"], y, xerr=plot_data["std"], fmt="o", capsize=4)
        ax.set_yticks(y, plot_data.index)
        ax.set_xlabel("Macro F1 medio ± desvío entre folds")
        ax.set_title("Pooling estático y estrategia de capas")
        ax.grid(axis="x", alpha=.25)
        plt.tight_layout()
        plt.show()

## Figura 2 — Capas y pooling: rendimiento frente a estabilidad

El cuadrante deseable está hacia la derecha y abajo. Los modelos entrenables se representan mediante su ensemble de semillas.

In [ ]:
if not results.empty:
    temporal = results.loc[
        (results["protocol"] == PROTOCOL_INDEPENDENT)
        & (results["target"] == TARGET_EMOTION_ORIGINAL)
        & (~results.get("result_type", pd.Series(index=results.index, dtype=str)).eq("seed"))
        & (results["representation"].astype(str).str.startswith("wav2vec"))
    ].copy()
    if temporal.empty:
        print("Sin configuraciones wav2vec para comparar.")
    else:
        plot_data = (
            temporal.groupby(["model", "refinement"], observed=True)["macro_f1"]
            .agg(["mean", "std"])
            .reset_index()
        )
        fig, ax = plt.subplots(figsize=(9, 5.5))
        for _, row in plot_data.iterrows():
            label = f"{row['model']} · {row['refinement']}"
            ax.scatter(row["mean"], row["std"], s=85)
            ax.annotate(label, (row["mean"], row["std"]), xytext=(5, 5), textcoords="offset points", fontsize=8)
        ax.set_xlabel("Macro F1 medio speaker-independent")
        ax.set_ylabel("Desvío estándar entre outer folds")
        ax.set_title("Rendimiento y estabilidad de representaciones wav2vec")
        ax.grid(alpha=.25)
        plt.tight_layout()
        plt.show()

## Figura 3 — Pesos de la mezcla aprendida

La curva permite observar si el modelo favorece capas tempranas, intermedias o profundas. La banda corresponde a la dispersión combinada entre folds y semillas, sin calcular índices de ranking.

In [ ]:
if layer_weights.empty:
    print("Sin pesos aprendidos de capas.")
else:
    subset = layer_weights.loc[
        (layer_weights["protocol"] == PROTOCOL_INDEPENDENT)
        & (layer_weights["target"] == TARGET_EMOTION_ORIGINAL)
    ]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for pooling, group in subset.groupby("pooling", observed=True):
        curve = group.groupby("layer", observed=True)["weight"].agg(["mean", "std"])
        x = curve.index.to_numpy()
        ax.plot(x, curve["mean"], marker="o", label=pooling)
        ax.fill_between(x, curve["mean"]-curve["std"], curve["mean"]+curve["std"], alpha=.18)
    ax.set_xlabel("Índice de capa (0 incluye feature projection si está configurada)")
    ax.set_ylabel("Peso softmax")
    ax.set_title("Mezcla escalar aprendida de capas wav2vec")
    ax.legend()
    ax.grid(alpha=.25)
    plt.tight_layout()
    plt.show()

## Figura 4 — Curvas por outer fold y variabilidad entre semillas

Las curvas muestran sensibilidad a los actores de validation. Para atención y mezcla aprendida, el panel derecho separa la variabilidad de inicialización.

In [ ]:
if not results.empty:
    primary = results.loc[
        (results["protocol"] == PROTOCOL_INDEPENDENT)
        & (results["target"] == TARGET_EMOTION_ORIGINAL)
    ].copy()
    ensemble = primary.loc[~primary.get("result_type", pd.Series(index=primary.index, dtype=str)).eq("seed")]
    seed_rows = primary.loc[primary.get("result_type", pd.Series(index=primary.index, dtype=str)).eq("seed")]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for (model, refinement), group in ensemble.groupby(["model", "refinement"], observed=True):
        if len(group) != int(cfg.splits.cv_folds):
            continue
        axes[0].plot(group.sort_values("fold")["fold"], group.sort_values("fold")["macro_f1"], marker="o", label=f"{model} · {refinement}")
    axes[0].set_title("Macro F1 por outer fold")
    axes[0].set_xlabel("Fold")
    axes[0].set_ylabel("Macro F1")
    axes[0].grid(alpha=.25)
    axes[0].legend(fontsize=7)

    if seed_rows.empty:
        axes[1].text(.5, .5, "Sin resultados por seed", ha="center", va="center")
        axes[1].set_axis_off()
    else:
        seed_summary = (
            seed_rows.groupby(["model", "refinement", "fold"], observed=True)["macro_f1"]
            .std(ddof=0)
            .reset_index(name="seed_std")
        )
        labels = seed_summary["model"] + " · " + seed_summary["refinement"]
        order = labels.drop_duplicates().tolist()
        values = [seed_summary.loc[labels == label, "seed_std"].to_numpy() for label in order]
        axes[1].boxplot(values, tick_labels=order, vert=True)
        axes[1].tick_params(axis="x", rotation=30)
        axes[1].set_ylabel("Desvío de macro F1 entre seeds")
        axes[1].set_title("Variabilidad de inicialización por fold")
        axes[1].grid(axis="y", alpha=.25)
    plt.tight_layout()
    plt.show()

## Figura 5 — Matrices OOF normalizadas

Se compara el baseline `last_mean` con la mejor configuración temporal disponible según macro F1 medio. La matriz diferencial muestra cambios en recall y desplazamientos de confusión.

In [ ]:
LABELS = ["angry", "calm", "disgust", "fearful", "happy", "neutral", "sad", "surprised"]

if oof_predictions.empty or results.empty:
    print("No hay predicciones OOF suficientes para matrices comparativas.")
else:
    candidates = results.loc[
        (results["protocol"] == PROTOCOL_INDEPENDENT)
        & (results["target"] == TARGET_EMOTION_ORIGINAL)
        & (~results.get("result_type", pd.Series(index=results.index, dtype=str)).eq("seed"))
        & (results["representation"].astype(str).str.startswith("wav2vec"))
    ]
    summary_candidates = (
        candidates.groupby(["model", "refinement"], observed=True)["macro_f1"]
        .mean()
        .sort_values(ascending=False)
    )
    baseline_key = ("logistic_regression", "last_mean")
    if baseline_key not in summary_candidates.index:
        print("Falta el baseline last_mean en temporal_oof_predictions.csv.")
    else:
        best_key = next((key for key in summary_candidates.index if key != baseline_key), baseline_key)
        def select_predictions(key):
            model, refinement = key
            return oof_predictions.loc[
                (oof_predictions["protocol"] == PROTOCOL_INDEPENDENT)
                & (oof_predictions["target"] == TARGET_EMOTION_ORIGINAL)
                & (oof_predictions["model"] == model)
                & (oof_predictions["refinement"] == refinement)
            ]
        base = select_predictions(baseline_key)
        best = select_predictions(best_key)
        if len(base) == 0 or len(best) == 0:
            print("Predicciones incompletas para la comparación.")
        else:
            cm_base = confusion_matrix(base["y_true"], base["y_pred"], labels=LABELS, normalize="true")
            cm_best = confusion_matrix(best["y_true"], best["y_pred"], labels=LABELS, normalize="true")
            matrices = [cm_base, cm_best, cm_best-cm_base]
            titles = ["Baseline: last mean", f"Mejor: {best_key[0]} · {best_key[1]}", "Cambio: mejor − baseline"]
            fig, axes = plt.subplots(1, 3, figsize=(20, 5.7))
            for index, (ax, matrix, title) in enumerate(zip(axes, matrices, titles)):
                cmap = "coolwarm" if index == 2 else "viridis"
                limit = np.abs(matrix).max() if index == 2 else None
                image = ax.imshow(matrix, cmap=cmap, vmin=-limit if index == 2 else 0, vmax=limit if index == 2 else 1)
                for row in range(len(LABELS)):
                    for col in range(len(LABELS)):
                        text = f"{matrix[row,col]:+.2f}" if index == 2 else f"{matrix[row,col]:.2f}"
                        ax.text(col, row, text, ha="center", va="center", fontsize=7)
                ax.set_xticks(range(len(LABELS)), LABELS, rotation=45, ha="right")
                ax.set_yticks(range(len(LABELS)), LABELS)
                ax.set_title(title)
                ax.set_xlabel("Predicción")
                if index == 0:
                    ax.set_ylabel("Clase verdadera")
                fig.colorbar(image, ax=ax, fraction=.046)
            plt.tight_layout()
            plt.show()

## Figura 6 — Atención temporal en ejemplos OOF

Se muestran hasta tres audios: un error corregido, un acierto compartido y un error persistente. Cuando el audio está disponible, el RMS se interpola únicamente como referencia visual.

In [ ]:
def load_attention_npz(path: Path) -> dict[str, np.ndarray]:
    if not path.exists():
        return {}
    with np.load(path, allow_pickle=False) as data:
        file_ids = data["file_ids"].astype(str)
        offsets = data["offsets"]
        values = data["values"]
    return {
        file_id: values[offsets[index]:offsets[index+1]]
        for index, file_id in enumerate(file_ids)
    }

attention_map = load_attention_npz(ATTENTION_WEIGHTS_PATH)
attention_pred = oof_predictions.loc[
    (oof_predictions.get("model") == "attentive_statistics")
    & (oof_predictions.get("protocol") == PROTOCOL_INDEPENDENT)
    & (oof_predictions.get("target") == TARGET_EMOTION_ORIGINAL)
] if not oof_predictions.empty else pd.DataFrame()
baseline_pred = oof_predictions.loc[
    (oof_predictions.get("model") == "logistic_regression")
    & (oof_predictions.get("refinement") == "last_mean")
    & (oof_predictions.get("protocol") == PROTOCOL_INDEPENDENT)
    & (oof_predictions.get("target") == TARGET_EMOTION_ORIGINAL)
] if not oof_predictions.empty else pd.DataFrame()

if not attention_map or attention_pred.empty or baseline_pred.empty:
    print("Sin pesos/predicciones de atención para visualizar ejemplos.")
else:
    comparison = baseline_pred[["file_id", "y_true", "y_pred"]].merge(
        attention_pred[["file_id", "y_pred"]], on="file_id", suffixes=("_base", "_att")
    )
    comparison["case"] = np.select(
        [
            (comparison["y_pred_base"] != comparison["y_true"]) & (comparison["y_pred_att"] == comparison["y_true"]),
            (comparison["y_pred_base"] == comparison["y_true"]) & (comparison["y_pred_att"] == comparison["y_true"]),
        ],
        ["Corregido por atención", "Acierto compartido"],
        default="Error persistente",
    )
    chosen = comparison.groupby("case", observed=True).head(1).head(3)
    meta_index = metadata.set_index("file_id")
    fig, axes = plt.subplots(len(chosen), 1, figsize=(12, 3.2*len(chosen)), squeeze=False)
    for ax, (_, row) in zip(axes[:,0], chosen.iterrows()):
        weights = attention_map[str(row["file_id"])]
        duration = float(meta_index.loc[row["file_id"], "duration_trimmed_s"])
        time = np.linspace(0, duration, len(weights))
        ax.plot(time, weights / max(weights.max(), 1e-8), label="Atención normalizada")
        audio_path = resolve_path(meta_index.loc[row["file_id"], "file_path_trimmed"])
        if audio_path.exists():
            import librosa
            waveform, sr = librosa.load(audio_path, sr=None, mono=True)
            rms = librosa.feature.rms(y=waveform)[0]
            rms_time = np.linspace(0, duration, len(rms))
            ax.plot(rms_time, rms / max(rms.max(), 1e-8), alpha=.65, label="RMS normalizado")
        ax.set_title(f"{row['case']} · {row['file_id']} · true={row['y_true']} · base={row['y_pred_base']} · att={row['y_pred_att']}")
        ax.set_xlabel("Tiempo [s]")
        ax.set_ylabel("Magnitud normalizada")
        ax.legend()
        ax.grid(alpha=.2)
    plt.tight_layout()
    plt.show()

## Figura 7 — Elastic Net e interpretación por familias eGeMAPS

El primer panel compara media y desvío de L2/Elastic Net. El segundo muestra cuánto peso y cuántos coeficientes no nulos conserva cada familia.

In [ ]:
if results.empty:
    print("Sin resultados eGeMAPS.")
else:
    eg_results = results.loc[
        (results["representation"] == "egemaps")
        & (results["protocol"] == PROTOCOL_INDEPENDENT)
        & (results["target"] == TARGET_EMOTION_ORIGINAL)
        & (results["refinement"].isin(["l2_reference", "elastic_net"]))
    ]
    if eg_results.empty:
        print("Sin comparación L2 vs Elastic Net.")
    else:
        performance = eg_results.groupby("refinement", observed=True)["macro_f1"].agg(["mean", "std"])
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        y = np.arange(len(performance))
        axes[0].errorbar(performance["mean"], y, xerr=performance["std"], fmt="o", capsize=4)
        axes[0].set_yticks(y, performance.index)
        axes[0].set_xlabel("Macro F1 medio ± desvío")
        axes[0].set_title("eGeMAPS: L2 vs Elastic Net")
        axes[0].grid(axis="x", alpha=.25)

        family = elastic_family.loc[
            (elastic_family["protocol"] == PROTOCOL_INDEPENDENT)
            & (elastic_family["target"] == TARGET_EMOTION_ORIGINAL)
        ] if not elastic_family.empty else pd.DataFrame()
        if family.empty:
            axes[1].text(.5, .5, "Sin importancia por familias", ha="center", va="center")
            axes[1].set_axis_off()
        else:
            family_summary = family.groupby("family_label", observed=True).agg(
                importance=("importance_normalized", "mean"),
                importance_std=("importance_normalized", "std"),
                nonzero=("nonzero_fraction", "mean"),
            ).sort_values("importance")
            y2 = np.arange(len(family_summary))
            axes[1].barh(y2, family_summary["importance"], xerr=family_summary["importance_std"], alpha=.75, label="Importancia")
            axes[1].scatter(family_summary["nonzero"], y2, marker="D", label="Proporción no nula")
            axes[1].set_yticks(y2, family_summary.index)
            axes[1].set_xlabel("Media entre folds")
            axes[1].set_title("Elastic Net por familia acústica")
            axes[1].legend()
            axes[1].grid(axis="x", alpha=.25)
        plt.tight_layout()
        plt.show()

## Figura final — Síntesis de rendimiento, estabilidad y dimensionalidad

El tamaño del punto representa la dimensionalidad global. La decisión debe considerar simultáneamente media, desvío, peor fold y variabilidad entre semillas; no solamente el mejor promedio.

In [ ]:
if not results.empty:
    primary = results.loc[
        (results["protocol"] == PROTOCOL_INDEPENDENT)
        & (results["target"] == TARGET_EMOTION_ORIGINAL)
        & (~results.get("result_type", pd.Series(index=results.index, dtype=str)).eq("seed"))
    ].copy()
    if primary.empty:
        print("Sin resultados primarios.")
    else:
        synthesis = primary.groupby(
            ["representation", "model", "refinement"], observed=True
        ).agg(
            macro_f1_mean=("macro_f1", "mean"),
            macro_f1_std=("macro_f1", "std"),
            worst_fold=("macro_f1", "min"),
            n_features=("n_features", "mean"),
        ).reset_index()
        fig, ax = plt.subplots(figsize=(11, 6))
        sizes = 45 + 190 * np.sqrt(synthesis["n_features"] / synthesis["n_features"].max())
        ax.scatter(synthesis["macro_f1_mean"], synthesis["macro_f1_std"], s=sizes, alpha=.75)
        for _, row in synthesis.iterrows():
            label = f"{row['model']} · {row['refinement']}"
            ax.annotate(label, (row["macro_f1_mean"], row["macro_f1_std"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
        ax.set_xlabel("Macro F1 medio speaker-independent")
        ax.set_ylabel("Desvío estándar entre folds")
        ax.set_title("Síntesis: rendimiento, estabilidad y dimensionalidad")
        ax.grid(alpha=.25)
        plt.tight_layout()
        plt.show()

        display(
            synthesis.sort_values(["macro_f1_mean", "macro_f1_std"], ascending=[False, True])
            .head(8)
            .round(3)
        )

## Criterio de cierre

Una estrategia se considera candidata si cumple al menos una condición:

- `Δ macro F1 medio ≥ 0.01`, o
- diferencia de media menor a `0.01` con reducción del desvío entre folds de al menos `20%`.

Además:

- ningún outer fold debe caer más de `0.03` frente a `last_mean`;
- en modelos entrenables, el desvío medio entre semillas debe ser menor a `0.02`;
- la conclusión debe indicar si la evidencia proviene de **variación temporal**, **capas intermedias**, **selección temporal aprendida** o **regularización acústica**.

La extensión speaker-adversarial queda fuera de este sprint y solo se justifica si la atención mejora frente a mean pooling pero la variabilidad sigue concentrada por actor.